In [42]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install transformers datasets
!pip install peft
!pip install evaluate scikit-learn transformers[torch]
!pip install accelerate>=0.26.0

Looking in indexes: https://download.pytorch.org/whl/cu121
ERROR: Could not find a version that satisfies the requirement torchaudio (from versions: none)
ERROR: No matching distribution found for torchaudio


# Prepare the dataset

In [43]:
from datasets import load_dataset

imdb = load_dataset("stanfordnlp/imdb")
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")


def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)


tokenized_datasets = imdb.map(tokenize_function, batched=True)


In [44]:
training_set = tokenized_datasets['train'].shuffle(42).select(range(5000))
shuffled_set =  tokenized_datasets['test'].shuffle(42)
validation_set  = shuffled_set.select(range(1000))
test_set = shuffled_set.select(range(1000, 2000))

## First method : Using the transformers library and pytorch


In [45]:
from transformers import BertTokenizer
from transformers import BertForSequenceClassification

## Fine-tuning the whole model

In [46]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForSequenceClassification.from_pretrained('bert-base-cased', num_labels=2)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5477.21it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint

In [47]:
# Have a look to 
#raise NotImplementedError

## LoRA implementation

In [48]:
[k for k,v  in model.bert.named_parameters()]

['embeddings.word_embeddings.weight',
 'embeddings.position_embeddings.weight',
 'embeddings.token_type_embeddings.weight',
 'embeddings.LayerNorm.weight',
 'embeddings.LayerNorm.bias',
 'encoder.layer.0.attention.self.query.weight',
 'encoder.layer.0.attention.self.query.bias',
 'encoder.layer.0.attention.self.key.weight',
 'encoder.layer.0.attention.self.key.bias',
 'encoder.layer.0.attention.self.value.weight',
 'encoder.layer.0.attention.self.value.bias',
 'encoder.layer.0.attention.output.dense.weight',
 'encoder.layer.0.attention.output.dense.bias',
 'encoder.layer.0.attention.output.LayerNorm.weight',
 'encoder.layer.0.attention.output.LayerNorm.bias',
 'encoder.layer.0.intermediate.dense.weight',
 'encoder.layer.0.intermediate.dense.bias',
 'encoder.layer.0.output.dense.weight',
 'encoder.layer.0.output.dense.bias',
 'encoder.layer.0.output.LayerNorm.weight',
 'encoder.layer.0.output.LayerNorm.bias',
 'encoder.layer.1.attention.self.query.weight',
 'encoder.layer.1.attention.se

In [49]:
print(f"What is the model \n\n {model}")


What is the model 

 BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNor

### Create the LoRA module

We will consider in this module the original linear, and the down and up projection

In [50]:
import torch 
import copy
from torch import nn

class LoRALinear(nn.Module):
  def __init__(
    self, in_dim: int, out_dim: int, rank: int
  ):
    super().__init__()

    #raise NotImplementedError

    self.linear = nn.Linear(in_dim, out_dim)            
    self.l_A = nn.Linear(in_dim, rank, bias=False)    
    self.l_B = nn.Linear(rank, out_dim, bias=False)   
    nn.init.zeros_(self.l_B.weight)                   

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    return self.linear(x) + self.l_B(self.l_A(x))

### Create a function replacing linear by lora module
From the original Linear module return a LoRA module
* The linear of the LoRA linear must be initialised with pretrained weights

In [51]:
def linear_to_lora(linear):
    linear_weight = linear.weight.data
    has_bias = linear.bias is not None
    if has_bias:
        linear_bias = linear.bias.data
    output_size, input_size =  linear_weight.shape
    lora = LoRALinear(input_size, output_size, rank=8)
    lora.linear.weight.data = linear_weight
    if has_bias:
        lora.linear.bias.data = linear_bias
    return lora

We now replace the target linear by the LoRALinear described above

In [52]:
lora_model = copy.deepcopy(model)
for block in lora_model.bert.encoder.layer:
    #raise NotImplementedError
    #print(block)
    
    lora = linear_to_lora(block.attention.self.query)
    block.attention.self.query = linear_to_lora(block.attention.self.query)
    block.attention.self.key = linear_to_lora(block.attention.self.key)
    block.attention.self.value = linear_to_lora(block.attention.self.value)

In [53]:
print(f"What is the model modified with LoRA: \n\n {lora_model}")


What is the model modified with LoRA: 

 BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): LoRALinear(
                (linear): Linear(in_features=768, out_features=768, bias=True)
                (l_A): Linear(in_features=768, out_features=8, bias=False)
                (l_B): Linear(in_features=8, out_features=768, bias=False)
              )
              (key): LoRALinear(
                (linear): Linear(in_features=768, out_features=768, bias=True)
                (l_A

### Defining module requiring grad

In [58]:
for k,v  in lora_model.bert.named_parameters():
    print(k)
    v.requires_grad = ('l_A' in k) or ('l_B' in k)

embeddings.word_embeddings.weight
embeddings.position_embeddings.weight
embeddings.token_type_embeddings.weight
embeddings.LayerNorm.weight
embeddings.LayerNorm.bias
encoder.layer.0.attention.self.query.linear.weight
encoder.layer.0.attention.self.query.linear.bias
encoder.layer.0.attention.self.query.l_A.weight
encoder.layer.0.attention.self.query.l_B.weight
encoder.layer.0.attention.self.key.linear.weight
encoder.layer.0.attention.self.key.linear.bias
encoder.layer.0.attention.self.key.l_A.weight
encoder.layer.0.attention.self.key.l_B.weight
encoder.layer.0.attention.self.value.linear.weight
encoder.layer.0.attention.self.value.linear.bias
encoder.layer.0.attention.self.value.l_A.weight
encoder.layer.0.attention.self.value.l_B.weight
encoder.layer.0.attention.output.dense.weight
encoder.layer.0.attention.output.dense.bias
encoder.layer.0.attention.output.LayerNorm.weight
encoder.layer.0.attention.output.LayerNorm.bias
encoder.layer.0.intermediate.dense.weight
encoder.layer.0.intermed

## Using transformers Trainer to fine-tune with LoRA 

In [59]:
import numpy as np
import evaluate

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [60]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(output_dir="test_custom_lora", 
                                  eval_strategy="steps",
                                  eval_steps= 128,
                                  num_train_epochs=2,)
trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset= training_set,
    eval_dataset=validation_set,
    compute_metrics=compute_metrics,
)


In [61]:
trainer.train()

Step,Training Loss,Validation Loss,Accuracy
128,No log,0.654552,0.631000
256,No log,0.581926,0.710000
384,No log,0.471865,0.771000
512,0.587049,0.368681,0.840000
640,0.587049,0.344428,0.860000
768,0.587049,0.359933,0.857000
896,0.587049,0.334816,0.862000
1024,0.348326,0.335543,0.865000
1152,0.348326,0.332909,0.868000
1250,0.348326,0.333702,0.867000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.60it/s]


TrainOutput(global_step=1250, training_loss=0.4362371765136719, metrics={'train_runtime': 1566.0245, 'train_samples_per_second': 6.386, 'train_steps_per_second': 0.798, 'total_flos': 2644700098560000.0, 'train_loss': 0.4362371765136719, 'epoch': 2.0})

In [62]:
lora_state = {
    k: v for k, v in lora_model.state_dict().items()
    if 'l_A' in k or 'l_B' in k or 'classifier' in k
}
torch.save(lora_state, "lora_weights.pth")

## Using PEFT library to fine-tune with LoRA 

In [54]:
from peft import LoraConfig, TaskType,  get_peft_model

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, r=1, lora_alpha=1, lora_dropout=0.1
)

model = BertForSequenceClassification.from_pretrained(
    'bert-base-cased', 
    num_labels=2
)
peft_model = get_peft_model(model, lora_config)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [56]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(output_dir="test_custom_lora", 
                                  eval_strategy="steps",
                                  eval_steps= 128,
                                  num_train_epochs=2,)
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset= training_set,
    eval_dataset=validation_set,
    compute_metrics=compute_metrics,
)

In [57]:
trainer.train()

Step,Training Loss,Validation Loss,Accuracy
128,No log,0.686927,0.548000
256,No log,0.677564,0.593000
384,No log,0.668519,0.616000
512,0.689200,0.655116,0.635000
640,0.689200,0.653552,0.616000
768,0.689200,0.630284,0.650000
896,0.689200,0.620816,0.659000
1024,0.652700,0.614406,0.658000
1152,0.652700,0.610755,0.665000


TrainOutput(global_step=1250, training_loss=0.6643443115234375, metrics={'train_runtime': 170.7475, 'train_samples_per_second': 58.566, 'train_steps_per_second': 7.321, 'total_flos': 2632290263040000.0, 'train_loss': 0.6643443115234375, 'epoch': 2.0})

## Compare the results of the different approaches